# Mesorregiões: identificação e suporte observado

# Cobertura regional de referências de custos

`identificacao_regional.csv` associa por código exato as 141 chaves do painel à DTB IBGE 2016 v2, com cinco mesorregiões. Os conjuntos de códigos coincidem integralmente. Há uma divergência nominal preservada: Santo Antônio **de** Leverger no painel e **do** Leverger na DTB. A conferência de chaves não valida estabilidade de limites entre 2016, 2023 e os anos históricos. Atribuições territoriais compostas e predecessores ainda exigem auditoria espacial; a tabela identifica referências de códigos, não harmoniza polígonos.

A região IMEA tem campo e vintage separados: nota de novembro de 2017, segmentação de 2008. São 140 correspondências de nome normalizado verificadas na origem. Poxoréu permanece NA; “Monte Verde do Norte” não ganhou código inventado. Boa Esperança do Norte não foi inserida retrospectivamente.

`referencias_identificadas.csv` tem 476 registros: 85 planos CONAB estritos, 195 colunas IMEA agrícolas e 196 pecuárias. Destes, 424 recebem mesorregião por nome normalizado exato. Os 52 restantes são 38 referências estaduais e 14 registros de abas pecuárias com identidade conflitante. Ausência de custo não é zero. Nenhum custo foi atribuído aos 141 municípios.

`cobertura_mesorregiao_atividade_ano.csv` contém também células de cobertura zero. Separa fonte, atividade e ano literal. Para CONAB, ano é data-base; para IMEA, é rótulo de origem, sem reinterpretação como ano de colheita. O período/safra original permanece na tabela de referências. IMEA milho de alta tecnologia não foi classificado automaticamente como primeira ou segunda safra. O diagnóstico conta referências, não soma COE/COT/CT como observações independentes. Os totais estaduais ficam fora das contagens mesorregionais.

A soja CONAB estrita cobre apenas Norte e Sudeste, com duas e uma praças, respectivamente; milho de primeira safra tem uma praça no Sudeste; milho de segunda safra tem duas praças em cada uma dessas mesorregiões. Não há validação preditiva mesorregional em grupos tão pequenos. Os testes agrícolas anteriores são entre praças e no tempo, não prova de transferência a uma região inteira. IMEA amplia a cobertura descritiva de praças, mas suas metodologias/unidades não foram fundidas à CONAB. Os dados suportam referências localizadas em regiões; não autorizam médias representativas, municipalização ou custos específicos de bioma.

Reproduzir: `python integrar.py --out /tmp/regioes_repro`, após executar ../pecuaria/integrar.py. Fontes compactas e proveniência original estão em fontes. Os raws IBGE/IMEA e scripts de extração geográfica estão no ambiente pai, não neste executor. Os artefatos com “pendente” e preparar.py documentam o estágio anterior, superado por integrar.py e manifesto_integracao.json; não devem ser usados como resultado atual.


In [1]:
from pathlib import Path
import csv, json
from integrar import run
report=run()
print(json.dumps(report,ensure_ascii=False,indent=2))

{
  "inputs": {
    "/workspace/tese_eco_usp/2_INTI/chapters/chap1/stats/modelos/base_custos_investigacao/regionalizacao/fontes/mt_regioes_141_compacto.csv": "9c125980a292bba5e8d899155b048a4521250c15d40f564d1f8806912d37c1b1",
    "/workspace/tese_eco_usp/2_INTI/chapters/chap1/stats/modelos/base3/painel_h1_base3_v4.csv": "a98dbe320efc784543cf43372a5903de32434be5871304178f2f9d49daaf1c3b",
    "/workspace/tese_eco_usp/2_INTI/chapters/chap1/stats/modelos/base_custos_investigacao/avaliacao_preditiva/soja/amostra.csv": "ea53e42f20b19096c6830ac4734500f218a137570704dfb671ec3671518ddae1",
    "/workspace/tese_eco_usp/2_INTI/chapters/chap1/stats/modelos/base_custos_investigacao/avaliacao_preditiva/milho_1a/amostra.csv": "4a74f06f6127443d5b122d41d03beb0571938e75f4cb3630fdf76ffcb316e8e6",
    "/workspace/tese_eco_usp/2_INTI/chapters/chap1/stats/modelos/base_custos_investigacao/avaliacao_preditiva/milho_2a/amostra.csv": "a0073c3f7a0c3ad8f4a3bb004887d1447582ec2a7edad8b7ffeff189b400f176",
    "/works

In [2]:
rows=list(csv.DictReader(Path('cobertura_mesorregiao_atividade_ano.csv').open()))
print('Linhas de cobertura:',len(rows))
for row in rows:
    if row['fonte']=='CONAB_estrita' and row['n_referencias_com_custos']!='0': print(row)

Linhas de cobertura: 535
{'fonte': 'CONAB_estrita', 'atividade': 'milho_1a', 'ano_rotulo': '2001', 'mesocodigo': '5105', 'mesonome': 'Sudeste Mato-grossense', 'vintage': 'DTB_2016_v2', 'n_referencias_periodo': '1', 'n_referencias_com_custos': '1', 'n_pracas_com_custos': '1', 'pracas': 'Primavera do Leste', 'modelo_regional_validado': 'False'}
{'fonte': 'CONAB_estrita', 'atividade': 'milho_1a', 'ano_rotulo': '2002', 'mesocodigo': '5105', 'mesonome': 'Sudeste Mato-grossense', 'vintage': 'DTB_2016_v2', 'n_referencias_periodo': '1', 'n_referencias_com_custos': '1', 'n_pracas_com_custos': '1', 'pracas': 'Primavera do Leste', 'modelo_regional_validado': 'False'}
{'fonte': 'CONAB_estrita', 'atividade': 'milho_1a', 'ano_rotulo': '2003', 'mesocodigo': '5105', 'mesonome': 'Sudeste Mato-grossense', 'vintage': 'DTB_2016_v2', 'n_referencias_periodo': '1', 'n_referencias_com_custos': '1', 'n_pracas_com_custos': '1', 'pracas': 'Primavera do Leste', 'modelo_regional_validado': 'False'}
{'fonte': 'CONA

In [3]:
import tempfile,hashlib
with tempfile.TemporaryDirectory() as d:
    reproduced=run(d)
    for n,h in reproduced['outputs'].items():
        assert hashlib.sha256(Path(n).read_bytes()).hexdigest()==h
print('PASS: reprodução isolada')

PASS: reprodução isolada
